# Notebook 02: Evaluation + Metrics (DCC26)

This notebook covers the evaluation segment (20 minutes):

1. Constraint checks for generated designs
2. Physics-based simulation via `problem.simulate`
3. Baseline comparison
4. Export metrics and plots


In [ ]:
# Colab/local dependency bootstrap
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
FORCE_INSTALL = False  # Set True to force reinstall outside Colab
PACKAGES = ['engibench[beams2d]', 'pandas', 'matplotlib']

if IN_COLAB or FORCE_INSTALL:
    print('Installing dependencies...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *PACKAGES])
    print('Dependency install complete.')
else:
    print('Skipping install (using current environment).')


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from engibench.problems.beams2d.v0 import Beams2D

ARTIFACT_DIR = Path('workshops/dcc26/artifacts')
assert ARTIFACT_DIR.exists(), 'Run Notebook 01 first (artifacts folder missing)'

gen_designs = np.load(ARTIFACT_DIR / 'generated_designs.npy')
baseline_designs = np.load(ARTIFACT_DIR / 'baseline_designs.npy')
with open(ARTIFACT_DIR / 'conditions.json', encoding='utf-8') as f:
    conditions = json.load(f)

print('generated:', gen_designs.shape)
print('baseline:', baseline_designs.shape)
print('conditions:', len(conditions))

In [ ]:
problem = Beams2D(seed=7)

rows = []
for i, (g, b, cfg) in enumerate(zip(gen_designs, baseline_designs, conditions, strict=True)):
    g_viol = problem.check_constraints(design=g, config=cfg)
    b_viol = problem.check_constraints(design=b, config=cfg)

    # Reset before simulator calls for reproducibility
    problem.reset(seed=7)
    g_obj = float(problem.simulate(g, config=cfg)[0])
    problem.reset(seed=7)
    b_obj = float(problem.simulate(b, config=cfg)[0])

    rows.append({
        'sample': i,
        'gen_obj': g_obj,
        'base_obj': b_obj,
        'gen_minus_base': g_obj - b_obj,
        'gen_violations': len(g_viol),
        'base_violations': len(b_viol),
    })

results = pd.DataFrame(rows)
results.head()

In [ ]:
def mean_pairwise_l2(designs: np.ndarray) -> float:
    flat = designs.reshape(designs.shape[0], -1)
    n = flat.shape[0]
    if n < 2:
        return 0.0
    dists = []
    for i in range(n):
        for j in range(i + 1, n):
            dists.append(float(np.linalg.norm(flat[i] - flat[j])))
    return float(np.mean(dists))

summary = {
    'n_samples': int(len(results)),
    'gen_obj_mean': float(results['gen_obj'].mean()),
    'base_obj_mean': float(results['base_obj'].mean()),
    'improvement_rate': float((results['gen_obj'] < results['base_obj']).mean()),
    'gen_violation_ratio': float((results['gen_violations'] > 0).mean()),
    'base_violation_ratio': float((results['base_violations'] > 0).mean()),
    'gen_diversity_l2': mean_pairwise_l2(gen_designs),
}

summary_df = pd.DataFrame([summary])
summary_df

In [ ]:
results.to_csv(ARTIFACT_DIR / 'per_sample_metrics.csv', index=False)
summary_df.to_csv(ARTIFACT_DIR / 'metrics_summary.csv', index=False)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(results['gen_obj'], bins=10, alpha=0.7, label='generated')
ax.hist(results['base_obj'], bins=10, alpha=0.7, label='baseline')
ax.set_xlabel('Compliance objective (lower is better)')
ax.set_ylabel('Count')
ax.set_title('Generated vs baseline objective distribution')
ax.legend()
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / 'objective_histogram.png', dpi=150)
plt.show()

print('Saved:')
print('-', ARTIFACT_DIR / 'per_sample_metrics.csv')
print('-', ARTIFACT_DIR / 'metrics_summary.csv')
print('-', ARTIFACT_DIR / 'objective_histogram.png')

In [ ]:
# Visual side-by-side sample grid
fig, axes = plt.subplots(3, 4, figsize=(12, 8))
for i, ax in enumerate(axes.ravel()):
    if i >= 12:
        break
    pair_idx = i // 2
    if i % 2 == 0:
        ax.imshow(gen_designs[pair_idx], cmap='gray', vmin=0, vmax=1)
        ax.set_title(f'gen {pair_idx}')
    else:
        ax.imshow(baseline_designs[pair_idx], cmap='gray', vmin=0, vmax=1)
        ax.set_title(f'base {pair_idx}')
    ax.axis('off')
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / 'design_grid.png', dpi=150)
plt.show()

## Interpretation hints

- `improvement_rate` shows how often generated designs beat baselines on objective value.
- `gen_violation_ratio` tracks practical feasibility pressure from constraints.
- `gen_diversity_l2` is a simple diversity proxy across generated designs.
